<a href="https://colab.research.google.com/github/raimund-buehler/visualizing-the-brain/blob/main/notebooks/day_two_de.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Tag Zwei in Colab öffnen"/></a>

# Tag Zwei: Stellt eure eigene Gehirnfrage

Heute führt eure Gruppe eine kleine fMRT-Analyse zu einer Frage durch, die ihr selbst auswählt.

Ihr werdet:

- zwei Aufgabenbedingungen auswählen und vergleichen,
- eine Vorhersage aufschreiben,
- ein First-Level-Modell anpassen,
- euren eigenen Kontrast berechnen,
- eine Gehirnkarte plotten, und
- erklären, was das Ergebnis bedeuten könnte.

Der Code ist weiterhin angeleitet. Eure Aufgabe ist es, die wissenschaftlichen Entscheidungen zu treffen und das Ergebnis vorsichtig zu interpretieren.

## Forschenden-Modus

Gestern haben wir ein Beispiel untersucht: rechte Tastendrücke im Vergleich zu linken Tastendrücken.

Heute wählt jede Gruppe eine eigene Frage. Zum Beispiel:

- **Vision:** Welche Areale reagieren stärker auf visuelle Muster?
- **Hören:** Welche Areale reagieren stärker, wenn eine Aufgabe gehört statt gesehen wird?
- **Sprache:** Welche Areale reagieren stärker auf Sätze als auf einfache visuelle Muster?
- **Rechen-Challenge:** Was verändert sich, wenn Menschen rechnen, statt Sätze zu verstehen?

Bevor ihr die Analyse ausführt, sollte sich eure Gruppe auf eine Frage und eine Vorhersage einigen.

## Setup

Führe diese Zelle zuerst aus. Sie installiert die nötigen Werkzeuge in Colab, importiert Python-Pakete, lädt die fMRT-Daten und lädt die anatomische Vorlage.

In [ ]:
# @title Setup: Werkzeuge installieren, Pakete importieren und Daten laden
import sys
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "nilearn", "nibabel", "matplotlib", "numpy", "pandas"
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import datasets, image, plotting
from nilearn.glm.first_level import FirstLevelModel

%matplotlib inline

anat_img = datasets.load_mni152_template(resolution=2)
fmri_dataset = datasets.fetch_localizer_first_level()
fmri_img = image.load_img(fmri_dataset.epi_img)
events = pd.read_table(fmri_dataset.events)

print("Setup abgeschlossen.")
print("fMRT-Bildform:", fmri_img.shape)
print("Anzahl der Ereignisse:", len(events))

## Wählt aus diesen Aufgabenbedingungen

Das Localizer-Experiment verwendet diese Bedingungsnamen. Ihr könnt sie genau so in die nächste Codezelle kopieren:

```text
horizontal_checkerboard
vertical_checkerboard
sentence_reading
sentence_listening
visual_computation
audio_computation
```

Die meisten Gruppen sollten mit Hören, Vision oder Sprache beginnen. Die Rechenbedingungen sind Challenge-Optionen, weil sie schwieriger zu interpretieren sein können.

## Wählt eure Forschungsfrage

Arbeitet in eurer Gruppe. Wählt eine Frage, bevor ihr den Code verändert.

Empfohlene Einstiegsfragen:

1. **Hören und Sprache:** `sentence_listening` verglichen mit `sentence_reading`.
2. **Vision:** `horizontal_checkerboard` verglichen mit `vertical_checkerboard`.
3. **Sprache:** `sentence_reading` verglichen mit `horizontal_checkerboard`.
4. **Sprache und Hören:** `sentence_listening` verglichen mit `horizontal_checkerboard`.
5. **Challenge:** `visual_computation` verglichen mit `sentence_reading`.
6. **Challenge:** `audio_computation` verglichen mit `sentence_listening`.
7. **Schwieriger:** `sentence_listening` verglichen mit `vertical_checkerboard`.

Ihr könnt Condition A und Condition B auch vertauschen. Dann stellt ihr die gegenteilige Frage, und warme und kühle Farben bedeuten das Gegenteil.

Manche Vergleiche sind leichter zu interpretieren als andere. Das ist in der fMRT-Forschung normal. Ein Kontrast kann einen klaren farbigen Cluster zeigen, aber der Grund für diesen Cluster kann trotzdem kompliziert sein.

Die Rechenbedingungen sind Challenge-Optionen. Sie können Areale zeigen, die mit Zahlen zu tun haben. Sie können aber auch Aufmerksamkeit, Anstrengung oder Schwierigkeit der Aufgabe zeigen. Interpretiert sie also nicht einfach als "das Mathe-Areal".

Schreibt eure Gruppenentscheidung hier auf:

- Wir wollen wissen, ob das Gehirn stärker reagiert auf: `__________`
- Verglichen mit: `__________`
- Wir erwarten stärkere Aktivität in der Nähe von: `__________`
- Wir erwarten das, weil: `__________`

## Mini-Aufgabe: Wählt Condition A und Condition B

Ein Kontrast vergleicht zwei Bedingungen:

```text
Condition A - Condition B
```

Warme Farben bedeuten: mehr Antwort für **Condition A** als für **Condition B**.

Kühle Farben bedeuten: mehr Antwort für **Condition B** als für **Condition A**.

Ändert in der nächsten Zelle nur die Bedingungsnamen in den Anführungszeichen.

In [ ]:
# Mini-Aufgabe: Wählt zwei Bedingungen.
# Kopiert die Bedingungsnamen genau und lasst die Anführungszeichen stehen.

condition_a = "sentence_listening"
condition_b = "sentence_reading"

my_contrast = condition_a + " - " + condition_b

print("Condition A:", condition_a)
print("Condition B:", condition_b)
print("Mein Kontrast:", my_contrast)

<details>
<summary>Braucht ihr Ideen? Klickt hier für Beispielauswahlen</summary>

```python
# Hören und Sprache: gehörte Sätze verglichen mit gelesenen Sätzen
condition_a = "sentence_listening"
condition_b = "sentence_reading"

# Vision: ein visuelles Muster verglichen mit einem anderen visuellen Muster
condition_a = "horizontal_checkerboard"
condition_b = "vertical_checkerboard"

# Sprache: Sätze lesen verglichen mit einem einfachen visuellen Muster
condition_a = "sentence_reading"
condition_b = "horizontal_checkerboard"

# Sprache und Hören: Sätze hören verglichen mit einem visuellen Muster
condition_a = "sentence_listening"
condition_b = "horizontal_checkerboard"

# Challenge: visuelle Rechenaufgaben verglichen mit Sätze lesen
# Das kann Zahlen betreffen, aber auch Aufmerksamkeit und Aufgabenschwierigkeit.
condition_a = "visual_computation"
condition_b = "sentence_reading"

# Challenge: gehörte Rechenaufgaben verglichen mit Sätze hören
# Das kann Zahlen betreffen, aber auch Aufmerksamkeit und Aufgabenschwierigkeit.
condition_a = "audio_computation"
condition_b = "sentence_listening"

# Schwieriger: Sätze hören verglichen mit einem anderen visuellen Muster
# Das mischt Hören/Sehen, Sprache und Unterschiede im visuellen Muster.
condition_a = "sentence_listening"
condition_b = "vertical_checkerboard"
```

Wählt ein Paar aus. Führt nicht alle Beispiele gleichzeitig aus.

</details>

## Stopp vor dem Plotten: macht eine Vorhersage

Geht nicht zu schnell über diesen Teil hinweg. Diskutiert in eurer Gruppe:

1. Was bedeuten warme Farben bei eurem Kontrast?
2. Was bedeuten kühle Farben?
3. Wo erwartet ihr die stärksten warmen Farben: vorne/hinten, links/rechts, oben/unten?
4. Vergleicht ihr einen klaren Unterschied, oder unterscheiden sich die Bedingungen in mehreren Dingen?

Schreibt einen Vorhersagesatz auf, bevor ihr weitermacht.

## Das Modell anpassen

Jetzt prüft Nilearn jedes Voxel. Für jedes Voxel fragt es, welche Aufgabenzeitpunkte hilfreich waren, um das fMRT-Signal dieses Voxels zu erklären.

Das kann einen kurzen Moment dauern. Diese Zelle müsst ihr nicht verändern.

In [ ]:
first_level_model = FirstLevelModel(
    t_r=2.4,
    slice_time_ref=0,
    hrf_model="spm",
    drift_model="cosine",
    high_pass=0.01,
    smoothing_fwhm=4,
    minimize_memory=True,
)

first_level_model = first_level_model.fit(fmri_img, events=events)
print("Modell erfolgreich angepasst.")

## Euren Kontrast berechnen

Jetzt fragen wir nach dem Vergleich, den ihr oben ausgewählt habt.

In [ ]:
my_map = first_level_model.compute_contrast(
    my_contrast,
    output_type="z_score",
)

print("Kontrast berechnet:", my_contrast)

## Eure Gehirnkarte plotten

Die Karte unten zeigt nur stärkere Unterschiede, mit einem Schwellenwert von `3.0`.

Denkt daran:

- Warme Farben bedeuten **Condition A > Condition B**.
- Kühle Farben bedeuten **Condition B > Condition A**.

Ihr könnt die Schichthöhen verändern, wenn euer Ergebnis höher oder tiefer im Gehirn liegt.

In [ ]:
slice_heights = [-10, 10, 30, 50, 70]

plotting.plot_stat_map(
    my_map,
    bg_img=anat_img,
    display_mode="z",
    cut_coords=slice_heights,
    threshold=3.0,
    symmetric_cbar=True,
    title=my_contrast,
)
plotting.show()

### Optional: interaktiv erkunden

Führt diese Zelle aus, wenn ihr im Ergebnis herumklicken möchtet.

In [ ]:
result_viewer = plotting.view_img(
    my_map,
    bg_img=anat_img,
    threshold=3.0,
    title=my_contrast,
)

result_viewer

## Euer Ergebnis interpretieren

Füllt das als Gruppe aus.

### Unser Kontrast

`__________ - __________`

### Was die Farben bedeuten

Warme Farben bedeuten mehr Antwort für: `__________`

Kühle Farben bedeuten mehr Antwort für: `__________`

### Wo wir Aktivität gefunden haben

Benutzt die Schichtansicht und die interaktive Ansicht, um die stärksten Cluster zu beschreiben. Ihr müsst nicht perfekt sein. Macht eine vorsichtige, gut begründete Vermutung.

Kurzer Lappen-Guide:

- **Okzipitallappen:** hinten im Gehirn, oft wichtig für Sehen.
- **Temporallappen:** seitlich/unterer Teil des Gehirns, oft wichtig für Hören und Sprache.
- **Parietallappen:** oben/hinten im Gehirn, oft wichtig für Aufmerksamkeit, Raum und zahlenbezogene Aufgaben.
- **Frontallappen:** vorne im Gehirn, oft wichtig für Planung, Aufmerksamkeit und schwierige Aufgaben.

Stärkste warme Farben:

- links / rechts / beide Seiten: `__________`
- vorne / Mitte / hinten: `__________`
- unten / Mitte / oben: `__________`
- wahrscheinlicher Lappen: `__________`
- Funktion, die zu diesem Lappen passen könnte: `__________`

Stärkste kühle Farben:

- links / rechts / beide Seiten: `__________`
- vorne / Mitte / hinten: `__________`
- unten / Mitte / oben: `__________`
- wahrscheinlicher Lappen: `__________`
- Funktion, die zu diesem Lappen passen könnte: `__________`

### Unsere Interpretation

Wir denken, dieses Ergebnis könnte bedeuten:

`__________`

### Eine Sache, bei der wir uns unsicher sind

`__________`

## Vorsichtig sein

Das sind echte fMRT-Daten, aber unsere Analyse ist vereinfacht.

- Wir haben eine Versuchsperson verwendet.
- Wir haben einen einfachen Darstellungs-Schwellenwert verwendet.
- Nicht jeder farbige Punkt ist automatisch eine Entdeckung.
- Manche Kontraste sind leichter zu interpretieren als andere.
- Ein Kontrast kann mehr als eine Sache gleichzeitig vergleichen.

Zum Beispiel hat `horizontal_checkerboard - vertical_checkerboard` eine recht klare Idee: Die beiden Aufgaben sind sehr ähnlich, aber das visuelle Muster verändert sich. `sentence_listening - sentence_reading` verändert dagegen mehr als eine Sache: Hören gegen Sehen, gesprochene Sprache gegen geschriebene Sprache und vielleicht Aufmerksamkeit. Die farbigen Cluster können trotzdem interessant sein, aber wir müssen vorsichtig sein, was wir behaupten.

Die Rechenkontraste sind besonders vorsichtige Entscheidungen. Wenn ihr `visual_computation - sentence_reading` vergleicht, könnte ein Cluster mit dem Arbeiten mit Zahlen zusammenhängen. Er könnte aber auch damit zusammenhängen, dass Rechnen schwieriger ist, mehr Aufmerksamkeit braucht oder Gedächtnis beansprucht. Deshalb sollten wir sagen: "Dieses Areal könnte an Berechnung beteiligt sein" und nicht: "Das ist das Mathe-Areal".

Das ist ein häufiges Problem in der fMRT-Forschung. Eine Gehirnkarte erklärt sich nicht selbst. Forschende müssen fragen: **Was genau haben wir verglichen? Was hat sich zwischen den beiden Bedingungen noch verändert? Könnte es eine andere Erklärung geben?**

Gute Wissenschaft bedeutet, sagen zu können, was ein Ergebnis nahelegt und worüber man noch unsicher ist.

## Bereitet eure zweiminütige Gruppenerklärung vor

Jede Gruppe sollte bereit sein zu erklären:

1. Unsere Forschungsfrage war...
2. Wir verglichen ... minus ...
3. Wir sagten voraus...
4. Warme Farben bedeuteten...
5. Kühle Farben bedeuteten...
6. Wir fanden...
7. Wir sind noch unsicher bei...

## Quellen

- Pinel, P., Thirion, B., Meriaux, S. et al. (2007). Fast reproducible identification and large-scale databasing of individual functional cognitive networks. *BMC Neuroscience, 8*, 91. https://doi.org/10.1186/1471-2202-8-91
- [Nilearn First-Level-Modelle](https://nilearn.github.io/stable/glm/first_level_model.html)
- [Nilearn-Plotting-Werkzeuge](https://nilearn.github.io/stable/modules/plotting.html)